# STEP 5, 6 & 7 — Baseline Selection & Parameter Derivation

## Objective
1.  **Select Neutral Baseline**: Choose the mode with the most balanced/stable profile from Step 3.
2.  **Derive Parameters**: Lock initial PCG settings based on this baseline's stats.
3.  **Transition**: Explicitly declare the end of calibration and start of training.


## Quick Start
Run the cell below to install dependencies.


In [1]:
# %pip install pandas numpy


In [2]:
import pandas as pd
import numpy as np
import os
import json

PROFILE_FILE = os.path.join('data', 'processed', 'mode_profiles.csv')
OUTPUT_PARAMS = os.path.join('config', 'initial_parameters.json')

if os.path.exists(PROFILE_FILE):
    df_profiles = pd.read_csv(PROFILE_FILE)
    print(f"Loaded Profiles for {df_profiles['modeId'].nunique()} modes.")
else:
    print("ERROR: Profiles not found. Run Notebook 02 first.")


Loaded Profiles for 3 modes.


## Step 5: Neutral Baseline Selection
**Criteria**:
1.  **Balanced Activity**: Presence of Combat, Exploration, and Collection (Low Sparsity across all dimensions).
2.  **Stability**: Lowest weighted variance.
3.  **Safety**: Death rate closest to 0 without being 0 (non-trivial).

We score each mode to suggest a candidate.


In [3]:
def score_baseline(df_profiles):
    modes = df_profiles['modeId'].unique()
    scores = []
    
    for mode in modes:
        stats = df_profiles[df_profiles['modeId'] == mode]
        
        # Criterion 1: Activity Balance (Avg Sparsity)
        mean_sparsity = stats['sparsity_pct'].mean()
        
        # Criterion 2: Stability (Avg Std Dev normalized by Mean? Simplification: Just Aggregated Std)
        # NOTE: This is a rough heuristic. Normalize first in real scenerio.
        mean_std = stats['std'].mean()
        
        # Criterion 3: Death Penalty (Too high is bad)
        death_row = stats[stats['metric'] == 'deathCountInWindow']
        deaths = death_row.iloc[0]['mean'] if not death_row.empty else 0
        
        # Simple Weighted Score (Lower is Better)
        # Heavy penalty for high sparsity (boredom) or high deaths (frustration)
        score = (mean_sparsity * 1.0) + (deaths * 100.0) + (mean_std * 0.1)
        
        scores.append({'modeId': mode, 'score': score, 'sparsity': mean_sparsity, 'deaths': deaths})
        
    return pd.DataFrame(scores).sort_values('score')

baseline_scores = score_baseline(df_profiles)
print("Baseline Candidate Ranking (Lower Score = More Neutral):")
print(baseline_scores)

selected_mode = baseline_scores.iloc[0]['modeId']
print(f"\n>> SELECTED BASELINE: {selected_mode}")


Baseline Candidate Ranking (Lower Score = More Neutral):
   modeId       score   sparsity    deaths
0       1   84.212439  48.148148  0.055556
1       2   86.000064  43.891403  0.110294
2       3  106.484247  50.310174  0.225806

>> SELECTED BASELINE: 1.0


## Step 6: Parameter Finalisation
We derive the initial parameters from the Selected Baseline's mean values.
Example Derivation: `InitialEnemyDensity = BaselineMean_EnemiesHit * 1.5` (Margin for challenge)


In [4]:
baseline_stats = df_profiles[df_profiles['modeId'] == selected_mode]
derived_params = {
    "_meta": {
        "source_mode": selected_mode,
        "derivation_method": "Mean + 20% Margin"
    }
}

for idx, row in baseline_stats.iterrows():
    metric = row['metric']
    val = row['mean']
    
    # Example derivation logic
    derived_params[f"target_{metric}"] = val * 1.2

# Ensure directory exists
os.makedirs('config', exist_ok=True)

with open(OUTPUT_PARAMS, 'w') as f:
    json.dump(derived_params, f, indent=2)

print(f"Final Parameters Saved to: {OUTPUT_PARAMS}")
print(json.dumps(derived_params, indent=2))


Final Parameters Saved to: config\initial_parameters.json
{
  "_meta": {
    "source_mode": 1.0,
    "derivation_method": "Mean + 20% Margin"
  },
  "target_enemiesHit": 3.7629629629629626,
  "target_damageDone": 53.82716049382716,
  "target_timeInCombat": 6.795865027634082,
  "target_deathOccurredInWindow": 0.0666666666666666,
  "target_deathCountInWindow": 0.0666666666666666,
  "target_distanceTraveled": 8716.24121873943,
  "target_timeSprinting": 5.248055382900767,
  "target_timeOutOfCombat": 29.43576223361823,
  "target_itemsCollected": 1.1481481481481481,
  "target_heartsCollected": 0.5703703703703703,
  "target_coinsCollected": 0.5777777777777776,
  "target_pickupAttempts": 4.207407407407407,
  "target_timeNearInteractables": 2.1370516873344227
}


## Step 7: Explicit Transition

> **DECLARATION**
> 
> The Calibration Phase is hereby **COMPLETE**.
> 
> *   Calibration Data (`calibration_dataset.csv`) is **FROZEN** and used ONLY for the derivations above.
> *   Initial Training Parameters are **LOCKED** in `config/initial_parameters.json`.
> *   Future Telemetry will be strictly for **MODEL TRAINING** (Adaptation).
> *   No feedback loop exists between Training Telemetry and these Calibration Parameters.
